# Baseline: минимальный пайплайн детекции дефектов

Берём данные **как есть** (без EDA и очистки), конвертируем в YOLO, обучаем, оцениваем.

**Порядок работы:**
1. Загрузка данных
2. ~~EDA~~ — пропускаем
3. ~~Очистка~~ — пропускаем
4. Конвертация в YOLO
5. Обучение модели
6. Оценка: mAP и FPS

## Импорты и настройки

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import shutil
import yaml
import time
import zipfile
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO

#указать путь где находится папка с репозиторием
DATA_DIR = Path(r"C:\code\neto")
# путь где находится папка с изображениями
TRAIN_IMAGES_DIR = DATA_DIR / "train_images"
# путь где будет находится сплит с йоло
YOLO_DIR = DATA_DIR / "yolo_dataset"
# ширина и высота изображений константна, в случае создания синтетики, можно ресайзить в этот размер
IMG_W, IMG_H = 1600, 256

## Готовые функции

Ячейку ниже нужно запустить один раз — она определяет все утилиты.

In [ ]:
def convert_to_yolo(df, image_dir, output_dir, img_w=1600, img_h=256):
    """Конвертирует DataFrame с bbox в структуру YOLO.

    Ожидает колонки: ImageId, ClassId, x_min, y_min, x_max, y_max, split.
    """
    if output_dir.exists():
        shutil.rmtree(output_dir)
    for split in ("train", "val"):
        (output_dir / "images" / split).mkdir(parents=True)
        (output_dir / "labels" / split).mkdir(parents=True)
    unique_classes = sorted(df["ClassId"].unique())
    class_map = {cls: i for i, cls in enumerate(unique_classes)}
    class_names = [f"defect_{cls}" for cls in unique_classes]
    copied, written = 0, 0
    for split in ("train", "val"):
        subset = df[df["split"] == split]
        for img_id, group in subset.groupby("ImageId"):
            src = image_dir / img_id
            dst = output_dir / "images" / split / img_id
            if src.exists() and not dst.exists():
                shutil.copy2(src, dst)
                copied += 1
            lines = []
            for _, row in group.iterrows():
                xc = ((row["x_min"] + row["x_max"]) / 2) / img_w
                yc = ((row["y_min"] + row["y_max"]) / 2) / img_h
                bw = (row["x_max"] - row["x_min"]) / img_w
                bh = (row["y_max"] - row["y_min"]) / img_h
                xc, yc = np.clip(xc, 0, 1), np.clip(yc, 0, 1)
                bw, bh = np.clip(bw, 0, 1), np.clip(bh, 0, 1)
                lines.append(f"{class_map[row['ClassId']]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
            label_path = output_dir / "labels" / split / (Path(img_id).stem + ".txt")
            label_path.write_text("\n".join(lines), encoding="utf-8")
            written += 1
    data_yaml = {
        "path": str(output_dir.resolve()), "train": "images/train",
        "val": "images/val", "nc": len(class_names), "names": class_names,
    }
    with open(output_dir / "data.yaml", "w", encoding="utf-8") as f:
        yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)
    print(f"Классы ({len(class_names)}): {class_names}")
    print(f"Скопировано изображений: {copied}, label-файлов: {written}")


def evaluate_model(model_path, data_yaml, img_dir=None, warmup=20):
    """Запускает валидацию модели (mAP) и замеряет FPS."""
    model = YOLO(model_path)
    print("=" * 50)
    print("Валидация (mAP)")
    print("=" * 50)
    metrics = model.val(data=data_yaml, verbose=False)
    print(f"mAP@0.5:      {metrics.box.map50:.4f}")
    print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
    print(f"Per-class AP@0.5: {dict(zip(metrics.names.values(), [round(v, 4) for v in metrics.box.ap50]))}")
    if img_dir is None:
        yaml_cfg = yaml.safe_load(Path(data_yaml).read_text(encoding="utf-8"))
        img_dir = Path(yaml_cfg["path"]) / yaml_cfg["val"]
    img_dir = Path(img_dir)
    images = sorted(img_dir.glob("*.jpg"))
    if not images:
        print(f"Нет изображений в {img_dir}")
        return
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\n{'=' * 50}")
    print(f"Замер FPS ({device}, batch=1, {len(images)} изображений, warmup={warmup})")
    print("=" * 50)
    for i in range(warmup):
        model.predict(str(images[i % len(images)]), verbose=False, device=device)
    times = []
    for img_path in images:
        t0 = time.perf_counter()
        model.predict(str(img_path), verbose=False, device=device)
        times.append(time.perf_counter() - t0)
    times = np.array(times)
    print(f"Среднее время на кадр: {times.mean()*1000:.1f} ms")
    print(f"FPS:                   {1/times.mean():.1f}")
    print(f"Требование 60 FPS:     {'PASS' if 1/times.mean() >= 60 else 'FAIL'}")


def export_for_cvat(yolo_dir, split="val", output_zip=None):
    """Собирает ZIP-датасет для импорта в CVAT (формат YOLO 1.1).

    Архив содержит изображения и аннотации — готов для
    CVAT → Create task → Import dataset → YOLO 1.1.
    """
    yolo_dir = Path(yolo_dir)
    labels_dir = yolo_dir / "labels" / split
    images_dir = yolo_dir / "images" / split
    with open(yolo_dir / "data.yaml", encoding="utf-8") as f:
        data_cfg = yaml.safe_load(f)
    class_names = data_cfg["names"]
    if output_zip is None:
        output_zip = yolo_dir.parent / f"cvat_{split}.zip"
    label_files = sorted(labels_dir.glob("*.txt"))
    with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("obj.names", "\n".join(class_names))
        obj_data = f"classes = {len(class_names)}\nnames = obj.names\ntrain = train.txt\nbackup = backup/\n"
        zf.writestr("obj.data", obj_data)
        train_txt = []
        for lf in label_files:
            zf.write(lf, f"obj_train_data/{lf.name}")
            img_path = images_dir / (lf.stem + ".jpg")
            if img_path.exists():
                zf.write(img_path, f"obj_train_data/{img_path.name}")
            train_txt.append(f"obj_train_data/{lf.stem}.jpg")
        zf.writestr("train.txt", "\n".join(train_txt))
    print(f"ZIP для CVAT: {output_zip} ({len(label_files)} файлов)")
    return output_zip


def import_from_cvat(zip_path, df, img_w=1600, img_h=256):
    """Импортирует исправленные аннотации из CVAT ZIP (YOLO 1.1) обратно в DataFrame.

    Для изображений, найденных в архиве, аннотации заменяются на новые.
    Остальные изображения в df остаются без изменений. Колонка split сохраняется.
    """
    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        obj_names = zf.read("obj.names").decode("utf-8").strip().split("\n")
        idx_to_class = {i: int(name.split("_")[1]) for i, name in enumerate(obj_names)}

        label_files = [f for f in zf.namelist()
                       if f.startswith("obj_train_data/") and f.endswith(".txt")]
        new_rows = []
        updated_images = set()
        for lf in label_files:
            img_id = Path(lf).stem + ".jpg"
            updated_images.add(img_id)
            content = zf.read(lf).decode("utf-8").strip()
            if not content:
                continue
            for line in content.split("\n"):
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                ci, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:])
                x_min = max(0, int(round((xc - bw / 2) * img_w)))
                y_min = max(0, int(round((yc - bh / 2) * img_h)))
                x_max = min(img_w, int(round((xc + bw / 2) * img_w)))
                y_max = min(img_h, int(round((yc + bh / 2) * img_h)))
                new_rows.append({
                    "ImageId": img_id, "ClassId": idx_to_class[ci],
                    "x_min": x_min, "y_min": y_min, "x_max": x_max, "y_max": y_max,
                })

    df_keep = df[~df["ImageId"].isin(updated_images)].copy()
    df_new = pd.DataFrame(new_rows)
    if "split" in df.columns and not df_new.empty:
        split_map = df.drop_duplicates("ImageId").set_index("ImageId")["split"]
        df_new["split"] = df_new["ImageId"].map(split_map)
    result = pd.concat([df_keep, df_new], ignore_index=True)
    print(f"Импорт из CVAT: {zip_path.name}")
    print(f"  Обновлено изображений: {len(updated_images)}")
    print(f"  Новых аннотаций:       {len(new_rows)}")
    print(f"  Итого в DataFrame:     {len(result)}")
    return result

print("Все функции загружены.")

---
## Шаг 1. Загрузка данных

In [3]:
df = pd.read_csv(DATA_DIR / "train_bboxes.csv")
print(f"Аннотаций: {len(df)}, изображений: {df['ImageId'].nunique()}")
print(f"Классы: {sorted(df['ClassId'].unique())}")
print(df["ClassId"].value_counts().sort_index())
print(f"\nСплит: {df['split'].value_counts().to_dict()}")
df.head()

Аннотаций: 7095, изображений: 6666
Классы: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
ClassId
1     601
2     247
3    2575
4     801
5    2575
6     296
Name: count, dtype: int64

Сплит: {'train': 5670, 'val': 1425}


,ImageId,ClassId,x_min,y_min,x_max,y_max,split
0,0002cc93b.jpg,1,113,63,755,255,train
1,0007a71bf.jpg,5,72,4,1167,255,train
2,000a4bcdd.jpg,1,146,55,461,255,train
3,000f6bf48.jpg,4,515,0,1131,255,train
4,0014fce06.jpg,3,896,60,929,249,train


---
## Шаг 2. EDA

В baseline пропускаем — берём данные как есть.

---
## Шаг 3. Очистка данных

В baseline пропускаем — никаких преобразований.

---
## Шаг 4. Конвертация в YOLO

In [4]:
print("Распределение классов по сплитам:")
print(df.groupby(["split", "ClassId"]).size().unstack(fill_value=0))

Распределение классов по сплитам:
ClassId    1    2     3    4     5    6
split                                  
train    475  214  2051  628  2074  228
val      126   33   524  173   501   68


In [5]:
convert_to_yolo(df, image_dir=TRAIN_IMAGES_DIR, output_dir=YOLO_DIR)

Классы (6): ['defect_1', 'defect_2', 'defect_3', 'defect_4', 'defect_5', 'defect_6']
Скопировано изображений: 6666, label-файлов: 6666


---
## Шаг 5. Обучение модели

In [6]:
model = YOLO("yolo11n.pt")  # nano — самая быстрая для прохода по бенчмарку
results = model.train(
    data=str(YOLO_DIR / "data.yaml"),
    epochs=25,
    imgsz=800, # оптимально сжать картинки вдвое тк не даст потери в точности, но позволит увлеичить батч
    batch=48,
    project=str(DATA_DIR / "runs"),
    name="baseline",
)

New https://pypi.org/project/ultralytics/8.4.56 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.55  Python-3.12.0 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=48, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\code\neto\yolo_dataset\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, mul

---
## Шаг 6. Оценка качества

Функция `evaluate_model` выведет mAP@0.5 и замерит FPS.
Минимальное требование: **60 FPS** на GPU.

In [ ]:
# при нескольких обучениях  можно просто указать путь до модели
evaluate_model(
    model_path=str(DATA_DIR / "runs" / "baseline-4" / "weights" / "best.pt"),
    data_yaml=str(YOLO_DIR / "data.yaml"),
)

Валидация (mAP)
Ultralytics 8.4.55  Python-3.12.0 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)


YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1082.3346.5 MB/s, size: 77.0 KB)
val: Scanning C:\code\neto\yolo_dataset\labels\val.cache... 1334 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1334/1334  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 84/84 17.2it/s 4.9s0.1s
                   all       1334       1425      0.455      0.587      0.518       0.33
Speed: 0.2ms preprocess, 0.7ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to C:\code\neto\runs\detect\val
mAP@0.5:      0.5182
mAP@0.5:0.95: 0.3304
Per-class AP@0.5: {'defect_1': np.float64(0.3298), 'defect_2': np.float64(0.4202), 'defect_3': np.float64(0.6445), 'defect_4': np.float64(0.7045), 'defect_5': np.float64(0.7963), 'defect_6': np.float64(0.214)}

Замер FPS (cuda, batch=1, 1334 изображений, warmup=20)
Среднее время на кадр: 9.5 ms
FPS:            

---
## Работа с CVAT (опционально)

Готовые функции позволяют визуально проверить и исправить аннотации в [CVAT](https://www.cvat.ai/).

**Экспорт → CVAT:**
1. Сконвертируйте данные в YOLO (`convert_to_yolo`).
2. Запустите `export_for_cvat` — она создаст ZIP с изображениями и аннотациями.
3. В CVAT: создайте проект → создайте задачу → **Actions → Import dataset → YOLO 1.1** → выберите ZIP.

**CVAT → Импорт обратно:**
1. Исправьте аннотации в CVAT.
2. Экспортируйте: **Actions → Export dataset → YOLO 1.1** → скачайте ZIP.
3. Запустите `import_from_cvat` — она обновит DataFrame исправленными аннотациями.
4. После этого можно заново запустить `convert_to_yolo` и обучение.

In [ ]:
# Экспорт в CVAT
# export_for_cvat(YOLO_DIR, split="val")

# Импорт исправленных аннотаций из CVAT
# df = import_from_cvat(DATA_DIR / "cvat_export.zip", df, img_w=IMG_W, img_h=IMG_H)
# df.to_csv(DATA_DIR / "train_bboxes.csv", index=False)  # сохранить обновлённый CSV
# convert_to_yolo(df, image_dir=TRAIN_IMAGES_DIR, output_dir=YOLO_DIR)  # пересоздать YOLO-датасет